# Phase 2 long-T step-10 reproducer (Colab A100)

Runs `experiments/phase2_step10_long_t.py` to sweep
T ∈ {10k, 100k, 1M, 10M} at N=5 seeds × 4 conditions, confirming
step 10's headline number holds at long training durations before
Block 9 commits N=500 cycles.

**Before running:** Runtime → Change runtime type → **A100 GPU**
(Colab Pro+). T=10M will fail on smaller GPUs.

**Expected wall-time:** ~90 min total at T=10M on A100. The
10k/100k/1M sweeps complete in the first few minutes; T=10M
dominates.

**Durability:** cell 3 mounts Google Drive, points
`overnight_results/` directly at
`MyDrive/silicritter_phase2/<UTC-timestamp>/`, **and seeds a test
write to fail-fast on any auth/permission/quota issue**. The
script writes CSV rows and log lines to Drive **as they are
produced**, not at end-of-run. If the Colab session times out
mid-run (close laptop, idle disconnect, etc.), partial state is
durable. Re-running the notebook detects the in-flight folder
and resumes from where it left off via the script's CSV-completed-
pairs check.

## 1. Confirm A100 attached

Should show `NVIDIA A100-SXM4-40GB` (or 80GB on some Pro+ tiers).
If it shows T4/V100/L4, change runtime type and re-execute.

In [ ]:
!nvidia-smi

## 2. Clone silicritter and install

Public repo, no auth needed. `pip install -e .` registers the
package in editable mode and pulls JAX with CUDA12 support.

In [ ]:
!git clone --depth 1 https://github.com/edhodapp/silicritter.git
%cd silicritter
!pip install -q -e ".[dev]"

## 3. Mount Drive, point `overnight_results/` at it, seed a test write

First run: Colab opens a popup to authorize Drive access for the
notebook (separate from any Claude MCP auth — Colab needs its
own grant on the same Google account). Subsequent runs reuse
the mount.

Resume detection: if the most recent `silicritter_phase2/<...>/`
folder has a CSV with fewer than 80 rows, that run was
interrupted and we resume into it (the script picks up where it
stopped via `_completed_pairs`). Otherwise we create a fresh
timestamped folder.

**Seed-write:** writes a tiny `.colab_write_seed.txt` to the
Drive folder and reads it back. Forces the auth/permission
round-trip to happen here (~1 second), not 90 minutes into the
Phase 2 run. If this cell errors, fix the Drive auth before
running cell 6.

In [ ]:
import csv
import shutil
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

EXPECTED_ROWS = 80  # 4 durations x 5 seeds x 4 conditions
base = Path('/content/drive/MyDrive/silicritter_phase2')
base.mkdir(parents=True, exist_ok=True)

def _row_count(folder: Path) -> int:
    csv_path = folder / 'phase2_step10_long_t.csv'
    if not csv_path.exists():
        return 0
    with csv_path.open() as f:
        return sum(1 for _ in csv.DictReader(f))

existing = sorted([p for p in base.iterdir() if p.is_dir()], reverse=True)
if existing and _row_count(existing[0]) < EXPECTED_ROWS:
    output_dir = existing[0]
    print(f'Resuming in-flight run ({_row_count(output_dir)}/{EXPECTED_ROWS} rows): {output_dir}')
else:
    stamp = datetime.now(tz=timezone.utc).strftime('%Y-%m-%dT%H%M%SZ')
    output_dir = base / stamp
    output_dir.mkdir()
    print(f'Starting new run: {output_dir}')

# Seed-write: force the Drive auth/permission round-trip NOW so any
# failure surfaces immediately, not 90 min into the run. drive.mount
# returning success doesn't guarantee writes work - first actual write
# is what hits the auth scope and quota path.
seed_path = output_dir / '.colab_write_seed.txt'
seed_text = f'seeded {datetime.now(tz=timezone.utc).isoformat()}\n'
seed_path.write_text(seed_text)
assert seed_path.read_text() == seed_text, 'Drive seed-write/read-back mismatch'
print(f'Drive write seed ok: {seed_path}')

# Symlink overnight_results/ -> Drive folder so the script writes
# directly to Drive as rows complete.
local = Path('overnight_results')
if local.is_symlink():
    local.unlink()
elif local.is_dir():
    shutil.rmtree(local)
local.symlink_to(output_dir)
print(f'overnight_results/ -> {output_dir}')

## 4. Verify JAX sees the A100

Should print `JAX backend: gpu` and a device list containing
`CudaDevice(id=0)` (or similar). If `cpu`, JAX didn't find the
GPU; check that the runtime type is set to A100.

In [ ]:
import jax
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")
assert jax.default_backend() == 'gpu', 'Need GPU runtime; switch via Runtime > Change runtime type'

## 5. Sanity-check the test suite

Runs the full pytest suite (~3 min on A100) before committing
to the 90-min Phase 2 run. If anything is broken in the Colab
environment (CUDA mismatch, JAX version drift, etc.), this
fails fast instead of mid-run.

In [ ]:
!python -m pytest tests/ -q --tb=short -x

## 6. Run Phase 2 (the big one)

Outputs land in the symlinked `overnight_results/` (i.e. directly
on Drive). `python -u` for unbuffered stdout so you see lines as
emitted. If the session times out mid-run, the partial CSV is
durable on Drive — re-run the notebook from cell 1 and cell 3 will
auto-resume into the same folder.

In [ ]:
!python -u experiments/phase2_step10_long_t.py

## 7. Display results

Files are already on Drive (cell 3 symlinked). This cell just
shows them inline so the kernel-state browser also has a copy.
Total ~12 KB.

In [ ]:
print('=' * 60)
print('CSV: overnight_results/phase2_step10_long_t.csv')
print(f'(symlinked to {output_dir})')
print('=' * 60)
!cat overnight_results/phase2_step10_long_t.csv
print()
print('=' * 60)
print('LOG: overnight_results/phase2_step10_long_t.log')
print('=' * 60)
!cat overnight_results/phase2_step10_long_t.log